# Monitoring Rules & Gold Alerts — Verification Suite (`tests/04_monitoring_rules_tests.ipynb`)

This notebook runs the synthetic boundary tests and detailed Gold read-back/rerun verification for `pipelines/04_monitoring_rules.ipynb`.

In [ ]:
# Step 1: Configuration and rule evaluation helper matching pipelines/04_monitoring_rules.ipynb
from decimal import Decimal
import os
import sys
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

for _candidate_dir in (
    os.getcwd(),
    os.path.join(os.getcwd(), "pipelines"),
    os.path.abspath(os.path.join(os.getcwd(), "..", "pipelines")),
):
    if os.path.isfile(os.path.join(_candidate_dir, "historical_features.py")) and _candidate_dir not in sys.path:
        sys.path.insert(0, _candidate_dir)

from historical_features import (
    FEATURE_COLUMNS,
    assert_feature_grain_and_null_semantics,
    compute_historical_features,
)

SILVER_TRANSACTIONS_TABLE = "workspace.sentinel_silver.transactions"
GOLD_SCHEMA = "workspace.sentinel_gold"
GOLD_ALERTS_TABLE = f"{GOLD_SCHEMA}.alerts"
ACTIVE_RULESET_VERSION = "ruleset-v1.0-provisional"

RULE_01_ID = "RULE_01_CARD_VELOCITY_1H"
RULE_01_VERSION = "v1.0-provisional"
RULE_01_WINDOW = "[event_time - 1h, event_time)"
RULE_01_MIN_PRIOR_CARD_TXNS_1H = 3

RULE_02_ID = "RULE_02_UNUSUAL_AMOUNT_14D"
RULE_02_VERSION = "v1.0-provisional"
RULE_02_WINDOW = "[event_time - 14d, event_time)"
RULE_02_MIN_PRIOR_CARD_TXNS_14D = 5
RULE_02_AMOUNT_MULTIPLIER = Decimal("3.00")

RULE_03_ID = "RULE_03_REPEATED_SIMILAR_MERCHANT_1H"
RULE_03_VERSION = "v1.0-provisional"
RULE_03_WINDOW = "[event_time - 1h, event_time)"
RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H = 2
RULE_03_AMOUNT_TOLERANCE_PCT = Decimal("0.10")

GOLD_ALERTS_COLUMNS = [
    "alert_id",
    "trans_num",
    "ruleset_version",
    "event_time",
    "cc_num",
    "merchant",
    "category",
    "amount",
    "replay_batch",
    "source_file",
    "source_row_index",
    "card_txn_count_1h",
    "card_txn_count_14d",
    "card_avg_amount_14d",
    "unusual_amount_threshold_14d",
    "card_merchant_txn_count_1h",
    "card_merchant_similar_amount_count_1h",
    "similar_amount_lower_bound",
    "similar_amount_upper_bound",
    "matched_rule_count",
    "matched_rule_ids",
    "matched_rule_versions",
    "rule_evidence",
]


def evaluate_monitoring_rules(feature_df: DataFrame) -> DataFrame:
    """Evaluate the three provisional rules and attach rule_evidence (1 row per trans_num)."""
    lower_factor = F.lit(Decimal("1.00") - RULE_03_AMOUNT_TOLERANCE_PCT)
    upper_factor = F.lit(Decimal("1.00") + RULE_03_AMOUNT_TOLERANCE_PCT)

    flagged_df = (
        feature_df.withColumn(
            "unusual_amount_threshold_14d",
            F.when(
                F.col("card_avg_amount_14d").isNotNull(),
                F.round(
                    F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER), 2
                ).cast("decimal(18,2)"),
            ).otherwise(F.lit(None).cast("decimal(18,2)")),
        )
        .withColumn(
            "similar_amount_lower_bound",
            F.round(F.col("amount") * lower_factor, 2).cast("decimal(18,2)"),
        )
        .withColumn(
            "similar_amount_upper_bound",
            F.round(F.col("amount") * upper_factor, 2).cast("decimal(18,2)"),
        )
        .withColumn(
            "rule_01_velocity_match",
            F.col("card_txn_count_1h") >= F.lit(RULE_01_MIN_PRIOR_CARD_TXNS_1H),
        )
        .withColumn(
            "rule_02_unusual_amount_match",
            (F.col("card_txn_count_14d") >= F.lit(RULE_02_MIN_PRIOR_CARD_TXNS_14D))
            & F.col("card_avg_amount_14d").isNotNull()
            & (
                F.col("amount")
                >= (F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER))
            ),
        )
        .withColumn(
            "rule_03_similar_merchant_match",
            F.col("card_merchant_similar_amount_count_1h")
            >= F.lit(RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H),
        )
    )

    rule_01_evidence = F.when(
        F.col("rule_01_velocity_match"),
        F.struct(
            F.lit(RULE_01_ID).alias("rule_id"),
            F.lit(RULE_01_VERSION).alias("rule_version"),
            F.lit(RULE_01_WINDOW).alias("window"),
            F.concat(
                F.lit("card_txn_count_1h="),
                F.col("card_txn_count_1h").cast("string"),
            ).alias("observed_metrics"),
            F.lit(f"card_txn_count_1h >= {RULE_01_MIN_PRIOR_CARD_TXNS_1H}").alias(
                "threshold_definition"
            ),
        ),
    )

    rule_02_evidence = F.when(
        F.col("rule_02_unusual_amount_match"),
        F.struct(
            F.lit(RULE_02_ID).alias("rule_id"),
            F.lit(RULE_02_VERSION).alias("rule_version"),
            F.lit(RULE_02_WINDOW).alias("window"),
            F.concat(
                F.lit("amount="),
                F.col("amount").cast("string"),
                F.lit(", card_avg_amount_14d="),
                F.col("card_avg_amount_14d").cast("string"),
                F.lit(", card_txn_count_14d="),
                F.col("card_txn_count_14d").cast("string"),
            ).alias("observed_metrics"),
            F.concat(
                F.lit(
                    f"card_txn_count_14d >= {RULE_02_MIN_PRIOR_CARD_TXNS_14D} AND amount >= {RULE_02_AMOUNT_MULTIPLIER} * card_avg_amount_14d ("
                ),
                F.col("unusual_amount_threshold_14d").cast("string"),
                F.lit(")"),
            ).alias("threshold_definition"),
        ),
    )

    rule_03_evidence = F.when(
        F.col("rule_03_similar_merchant_match"),
        F.struct(
            F.lit(RULE_03_ID).alias("rule_id"),
            F.lit(RULE_03_VERSION).alias("rule_version"),
            F.lit(RULE_03_WINDOW).alias("window"),
            F.concat(
                F.lit("amount="),
                F.col("amount").cast("string"),
                F.lit(", card_merchant_similar_amount_count_1h="),
                F.col("card_merchant_similar_amount_count_1h").cast("string"),
                F.lit(", card_merchant_txn_count_1h="),
                F.col("card_merchant_txn_count_1h").cast("string"),
            ).alias("observed_metrics"),
            F.concat(
                F.lit(
                    f"card_merchant_similar_amount_count_1h >= {RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H} within +/-10% ["
                ),
                F.col("similar_amount_lower_bound").cast("string"),
                F.lit(", "),
                F.col("similar_amount_upper_bound").cast("string"),
                F.lit("]"),
            ).alias("threshold_definition"),
        ),
    )

    filtered_evidence = F.filter(
        F.array(rule_01_evidence, rule_02_evidence, rule_03_evidence),
        lambda item: item.isNotNull(),
    )

    return (
        flagged_df.withColumn("ruleset_version", F.lit(ACTIVE_RULESET_VERSION))
        .withColumn("rule_evidence", filtered_evidence)
        .withColumn(
            "matched_rule_ids",
            F.transform(F.col("rule_evidence"), lambda item: item["rule_id"]),
        )
        .withColumn(
            "matched_rule_versions",
            F.transform(F.col("rule_evidence"), lambda item: item["rule_version"]),
        )
        .withColumn("matched_rule_count", F.size(F.col("matched_rule_ids")))
        .withColumn("is_flagged", F.col("matched_rule_count") > 0)
    )

### Step 2: Synthetic Rule & Alert Projection Tests

Verifies matches, non-matches, exact thresholds (`3` in 1h, `5` in 14d at `3.00x`, `2` in 1h within `±10%`), time boundaries (`-1h`, `-14d`, `-1h - 1s`, `-14d - 1s`), equal-timestamp exclusion, insufficient history, and multi-rule single-row grain.

In [ ]:
# Step 2: Execute synthetic tests for matches, non-matches, exact boundaries, and insufficient history
def assert_rule_evaluation_grain(
    input_df: DataFrame, evaluated_df: DataFrame
) -> dict[str, int]:
    """Verify 1:1 transaction grain and rule-match summary metrics."""
    summary = (
        evaluated_df.agg(
            F.count("*").alias("evaluated_rows"),
            F.countDistinct("trans_num").alias("distinct_trans_num"),
            F.sum(F.when(F.col("is_flagged"), 1).otherwise(0)).alias("flagged_transactions"),
            F.sum(F.when(F.col("rule_01_velocity_match"), 1).otherwise(0)).alias("rule_01_matches"),
            F.sum(F.when(F.col("rule_02_unusual_amount_match"), 1).otherwise(0)).alias("rule_02_matches"),
            F.sum(F.when(F.col("rule_03_similar_merchant_match"), 1).otherwise(0)).alias("rule_03_matches"),
            F.sum(F.when(F.col("matched_rule_count") > 1, 1).otherwise(0)).alias("multi_rule_transactions"),
        )
        .collect()[0]
        .asDict()
    )
    input_rows = input_df.count()
    assert summary["evaluated_rows"] == input_rows, f"Row count mismatch: {summary['evaluated_rows']} != {input_rows}"
    assert summary["distinct_trans_num"] == input_rows, "Duplicate trans_num in evaluated output"
    return {k: int(v or 0) for k, v in summary.items()}


synthetic_rows = [
    # --- Card R1: Velocity (Rule 1) boundary & equal-timestamp tests ---
    ("r1_tx1_no_hist",       "card_R1", "M_A", "misc", Decimal("100.00"), "2019-01-15 11:00:00"),
    ("r1_tx2_1_prior",       "card_R1", "M_B", "misc", Decimal("90.00"),  "2019-01-15 11:20:00"),
    ("r1_tx3_2_prior",       "card_R1", "M_C", "misc", Decimal("80.00"),  "2019-01-15 11:40:00"),
    ("r1_tx4_exact_1h_3",    "card_R1", "M_D", "misc", Decimal("70.00"),  "2019-01-15 12:00:00"),
    ("r1_tx5_eq_ts_a",       "card_R1", "M_E", "misc", Decimal("60.00"),  "2019-01-15 12:20:01"),
    ("r1_tx6_eq_ts_b",       "card_R1", "M_F", "misc", Decimal("50.00"),  "2019-01-15 12:20:01"),

    # --- Card R2: Unusual Amount (Rule 2) insufficient history (<5 prior payments) ---
    ("r2_tx1_exact_14d",     "card_R2", "M_A", "misc", Decimal("20.00"),  "2019-01-01 12:00:00"),
    ("r2_tx2_hist_2",        "card_R2", "M_B", "misc", Decimal("20.00"),  "2019-01-03 12:00:00"),
    ("r2_tx3_hist_3",        "card_R2", "M_C", "misc", Decimal("20.00"),  "2019-01-05 12:00:00"),
    ("r2_tx4_hist_4",        "card_R2", "M_D", "misc", Decimal("20.00"),  "2019-01-07 12:00:00"),
    ("r2_tx5_only_4_prior",  "card_R2", "M_E", "misc", Decimal("100.00"), "2019-01-09 12:00:00"),

    # --- Card R2B: 5 prior payments, amount 1 cent below 3x threshold (59.99 < 60.00) ---
    ("r2b_tx1_exact_14d",    "card_R2B", "M_A", "misc", Decimal("20.00"), "2019-01-01 12:00:00"),
    ("r2b_tx2_hist_2",       "card_R2B", "M_B", "misc", Decimal("20.00"), "2019-01-03 12:00:00"),
    ("r2b_tx3_hist_3",       "card_R2B", "M_C", "misc", Decimal("20.00"), "2019-01-05 12:00:00"),
    ("r2b_tx4_hist_4",       "card_R2B", "M_D", "misc", Decimal("20.00"), "2019-01-07 12:00:00"),
    ("r2b_tx5_hist_5",       "card_R2B", "M_E", "misc", Decimal("20.00"), "2019-01-09 12:00:00"),
    ("r2b_tx6_below_3x",     "card_R2B", "M_F", "misc", Decimal("59.99"), "2019-01-14 12:00:00"),

    # --- Card R2C: Exact 5 prior payments (including exact -14d) and exact 3x amount (60.00) ---
    ("r2c_tx1_exact_14d",    "card_R2C", "M_A", "misc", Decimal("20.00"), "2019-01-01 12:00:00"),
    ("r2c_tx2_hist_2",       "card_R2C", "M_B", "misc", Decimal("20.00"), "2019-01-03 12:00:00"),
    ("r2c_tx3_hist_3",       "card_R2C", "M_C", "misc", Decimal("20.00"), "2019-01-05 12:00:00"),
    ("r2c_tx4_hist_4",       "card_R2C", "M_D", "misc", Decimal("20.00"), "2019-01-07 12:00:00"),
    ("r2c_tx5_hist_5",       "card_R2C", "M_E", "misc", Decimal("20.00"), "2019-01-09 12:00:00"),
    ("r2c_tx6_exact_3x",     "card_R2C", "M_F", "misc", Decimal("60.00"), "2019-01-15 12:00:00"),

    # --- Card R2D: Earliest of 5 prior payments falls at -14d - 1s (excluded -> count=4 < 5) ---
    ("r2d_tx1_before_14d",   "card_R2D", "M_A", "misc", Decimal("20.00"), "2019-01-01 11:59:59"),
    ("r2d_tx2_hist_2",       "card_R2D", "M_B", "misc", Decimal("20.00"), "2019-01-03 12:00:00"),
    ("r2d_tx3_hist_3",       "card_R2D", "M_C", "misc", Decimal("20.00"), "2019-01-05 12:00:00"),
    ("r2d_tx4_hist_4",       "card_R2D", "M_D", "misc", Decimal("20.00"), "2019-01-07 12:00:00"),
    ("r2d_tx5_hist_5",       "card_R2D", "M_E", "misc", Decimal("20.00"), "2019-01-09 12:00:00"),
    ("r2d_tx6_dropped_14d",  "card_R2D", "M_F", "misc", Decimal("60.00"), "2019-01-15 12:00:00"),

    # --- Card R3A: Merchant count >= 2 with dissimilar amounts does NOT match Rule 3 ---
    ("r3a_tx1_dissim_low",   "card_R3A", "M_SIM", "retail", Decimal("50.00"),  "2019-01-15 11:10:00"),
    ("r3a_tx2_dissim_high",  "card_R3A", "M_SIM", "retail", Decimal("110.01"), "2019-01-15 11:20:00"),
    ("r3a_tx3_merch2_sim0",  "card_R3A", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 11:30:00"),

    # --- Card R3B: Exact +/-10% boundaries at exact -1h (Rule 3 only), then multi-rule (Rule 1 + Rule 3) ---
    ("r3b_tx1_exact_m10pct", "card_R3B", "M_SIM", "retail", Decimal("90.00"),  "2019-01-15 11:00:00"),
    ("r3b_tx2_exact_p10pct", "card_R3B", "M_SIM", "retail", Decimal("110.00"), "2019-01-15 11:30:00"),
    ("r3b_tx3_exact_sim2",   "card_R3B", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 12:00:00"),
    ("r3b_tx3b_other_merch", "card_R3B", "M_OTHER", "gas",  Decimal("25.00"),  "2019-01-15 12:05:00"),
    ("r3b_tx4_multi_r1_r3",  "card_R3B", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 12:10:00"),

    # --- Card R3C: -1h - 1s lower boundary exclusion for Rule 3 ---
    ("r3c_tx1_before_1h",    "card_R3C", "M_SIM", "retail", Decimal("90.00"),  "2019-01-15 10:59:59"),
    ("r3c_tx2_in_1h",        "card_R3C", "M_SIM", "retail", Decimal("110.00"), "2019-01-15 11:30:00"),
    ("r3c_tx3_dropped_1h",   "card_R3C", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 12:00:00"),
]

synthetic_input_df = (
    spark.createDataFrame(
        synthetic_rows,
        schema=["trans_num", "cc_num", "merchant", "category", "amount", "event_time_str"],
    )
    .withColumn("event_time", F.to_timestamp_ntz("event_time_str"))
    .withColumn("replay_batch", F.to_date(F.col("event_time")))
    .withColumn("source_file", F.lit("synthetic_fixture"))
    .withColumn("source_row_index", F.lit("0"))
    .drop("event_time_str")
)

synthetic_features_df = compute_historical_features(
    synthetic_input_df,
    similar_amount_tolerance_pct=RULE_03_AMOUNT_TOLERANCE_PCT,
)
assert_feature_grain_and_null_semantics(synthetic_input_df, synthetic_features_df)

synthetic_evaluated_df = evaluate_monitoring_rules(synthetic_features_df)
synthetic_summary = assert_rule_evaluation_grain(synthetic_input_df, synthetic_evaluated_df)

actual_outcomes = {
    row["trans_num"]: (
        row["card_txn_count_1h"],
        row["card_txn_count_14d"],
        row["card_avg_amount_14d"],
        row["card_merchant_txn_count_1h"],
        row["card_merchant_similar_amount_count_1h"],
        row["matched_rule_ids"],
    )
    for row in synthetic_evaluated_df.collect()
}

expected_outcomes = {
    "r1_tx1_no_hist":       (0, 0, None,              0, 0, []),
    "r1_tx2_1_prior":       (1, 1, Decimal("100.00"), 0, 0, []),
    "r1_tx3_2_prior":       (2, 2, Decimal("95.00"),  0, 0, []),
    "r1_tx4_exact_1h_3":    (3, 3, Decimal("90.00"),  0, 0, [RULE_01_ID]),
    "r1_tx5_eq_ts_a":       (2, 4, Decimal("85.00"),  0, 0, []),
    "r1_tx6_eq_ts_b":       (2, 4, Decimal("85.00"),  0, 0, []),
    "r2_tx5_only_4_prior":  (0, 4, Decimal("20.00"),  0, 0, []),
    "r2b_tx6_below_3x":     (0, 5, Decimal("20.00"),  0, 0, []),
    "r2c_tx6_exact_3x":     (0, 5, Decimal("20.00"),  0, 0, [RULE_02_ID]),
    "r2d_tx6_dropped_14d":  (0, 4, Decimal("20.00"),  0, 0, []),
    "r3a_tx3_merch2_sim0":  (2, 2, Decimal("80.01"),  2, 0, []),
    "r3b_tx1_exact_m10pct": (0, 0, None,              0, 0, []),
    "r3b_tx2_exact_p10pct": (1, 1, Decimal("90.00"),  1, 0, []),
    "r3b_tx3_exact_sim2":   (2, 2, Decimal("100.00"), 2, 2, [RULE_03_ID]),
    "r3b_tx3b_other_merch": (2, 3, Decimal("100.00"), 0, 0, []),
    "r3b_tx4_multi_r1_r3":  (3, 4, Decimal("81.25"),  2, 2, [RULE_01_ID, RULE_03_ID]),
    "r3c_tx3_dropped_1h":   (1, 2, Decimal("100.00"), 1, 1, []),
}

for tx_id, expected_tuple in expected_outcomes.items():
    actual_tuple = actual_outcomes.get(tx_id)
    if actual_tuple != expected_tuple:
        raise AssertionError(
            f"Synthetic rule test failed for {tx_id}: expected {expected_tuple}, got {actual_tuple}"
        )

synthetic_gold_df = (
    synthetic_evaluated_df.filter(F.col("is_flagged"))
    .withColumn(
        "alert_id",
        F.sha2(F.concat_ws("|", F.col("ruleset_version"), F.col("trans_num")), 256),
    )
    .select(*GOLD_ALERTS_COLUMNS)
)
assert synthetic_gold_df.columns == GOLD_ALERTS_COLUMNS
assert synthetic_gold_df.count() == 4
assert synthetic_gold_df.select("alert_id").distinct().count() == 4

print("All synthetic rule and Gold-projection tests passed.")
print("Synthetic rule summary:", synthetic_summary)
display(synthetic_gold_df.orderBy("event_time", "trans_num"))

### Step 3: Detailed Gold Read-Back, Synthetic-Isolation & Rerun Check

Verifies `workspace.sentinel_gold.alerts` against expected real-data alerts (`exceptAll == 0` in both directions), confirms zero synthetic fixture rows exist in Gold, and checks rerun idempotency.

In [ ]:
# Step 3: Detailed read-back, synthetic-isolation, and rerun idempotency verification on Gold alerts
SYNTHETIC_TEST_IDS = [row[0] for row in synthetic_rows]

real_silver_df = spark.read.table(SILVER_TRANSACTIONS_TABLE)
real_features_df = compute_historical_features(
    real_silver_df,
    similar_amount_tolerance_pct=RULE_03_AMOUNT_TOLERANCE_PCT,
)
assert_feature_grain_and_null_semantics(real_silver_df, real_features_df)

real_evaluated_df = evaluate_monitoring_rules(real_features_df)
real_eval_summary = assert_rule_evaluation_grain(real_silver_df, real_evaluated_df)
expected_real_gold_df = (
    real_evaluated_df.filter(F.col("is_flagged"))
    .withColumn(
        "alert_id",
        F.sha2(F.concat_ws("|", F.col("ruleset_version"), F.col("trans_num")), 256),
    )
    .select(*GOLD_ALERTS_COLUMNS)
)


def verify_gold_table_detailed(
    expected_df: DataFrame, expected_flagged_count: int
) -> DataFrame:
    saved_df = spark.read.table(GOLD_ALERTS_TABLE)
    assert saved_df.columns == GOLD_ALERTS_COLUMNS, f"Schema mismatch: {saved_df.columns}"
    assert "is_fraud" not in saved_df.columns, "'is_fraud' must not exist in Gold alerts"

    checks = (
        saved_df.agg(
            F.count("*").alias("saved_rows"),
            F.countDistinct("alert_id").alias("distinct_alert_ids"),
            F.countDistinct("trans_num").alias("distinct_trans_nums"),
            F.sum(
                F.when(F.col("trans_num").isin(SYNTHETIC_TEST_IDS), 1).otherwise(0)
            ).alias("synthetic_rows_in_gold"),
            F.sum(
                F.when(
                    F.col("alert_id").isNull()
                    | F.col("trans_num").isNull()
                    | F.col("rule_evidence").isNull()
                    | (F.col("matched_rule_count") < 1)
                    | (F.size(F.col("matched_rule_ids")) != F.col("matched_rule_count"))
                    | (F.size(F.col("matched_rule_versions")) != F.col("matched_rule_count"))
                    | (F.size(F.col("rule_evidence")) != F.col("matched_rule_count"))
                    | F.exists(
                        F.col("rule_evidence"),
                        lambda ev: ev["rule_id"].isNull()
                        | ev["rule_version"].isNull()
                        | ev["observed_metrics"].isNull()
                        | ev["threshold_definition"].isNull(),
                    ),
                    1,
                ).otherwise(0)
            ).alias("invalid_evidence_or_id_rows"),
        )
        .collect()[0]
        .asDict()
    )

    assert checks["saved_rows"] == expected_flagged_count
    assert checks["distinct_alert_ids"] == expected_flagged_count
    assert checks["distinct_trans_nums"] == expected_flagged_count
    assert int(checks["synthetic_rows_in_gold"] or 0) == 0, "Synthetic test rows found in Gold!"
    assert int(checks["invalid_evidence_or_id_rows"] or 0) == 0, "Invalid evidence or IDs in Gold!"

    assert expected_df.exceptAll(saved_df).count() == 0, "expected_df minus saved_df != 0"
    assert saved_df.exceptAll(expected_df).count() == 0, "saved_df minus expected_df != 0"
    return saved_df


first_saved_df = verify_gold_table_detailed(
    expected_real_gold_df,
    real_eval_summary["flagged_transactions"],
)
first_snapshot_df = spark.createDataFrame(
    first_saved_df.collect(),
    schema=first_saved_df.schema,
)

# Rerun overwrite and verify identical persisted output
expected_real_gold_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(GOLD_ALERTS_TABLE)

second_saved_df = verify_gold_table_detailed(
    expected_real_gold_df,
    real_eval_summary["flagged_transactions"],
)
assert first_snapshot_df.exceptAll(second_saved_df).count() == 0
assert second_saved_df.exceptAll(first_snapshot_df).count() == 0

print(
    f"Detailed Gold verification & rerun check passed: {second_saved_df.count():,} alerts "
    "(0 synthetic rows, two-way exceptAll == 0, identical rerun snapshot)."
)